# HiRAG-Ontology: Multi-Agent Knowledge Graph Construction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ekaesha/hirag-ontology/blob/main/hirag_ontology_colab.ipynb)

**Bachelor's Thesis — HSE DSBA 2026**  
Authors: Eva Karimova, Alexey Popov

This notebook demonstrates the full multi-agent pipeline:
- A1: Knowledge extraction from clinical guidelines
- A2: Ontological entity typing
- A3: Hybrid entity deduplication
- A4: Consistency validation
- A5: Missing relation inference
- A6: Knowledge graph update
- Evaluation: Hybrid RRF retrieval vs baselines

**Dataset:** Minzdrav oncological clinical guidelines (78 documents)

In [ ]:
import os
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TRANSFORMERS_OFFLINE"] = "0"
print("Offline mode disabled")
# Install dependencies
!pip install sentence-transformers rank-bm25 networkx openai python-dotenv -q

# Clone repository
!git clone https://github.com/ekaesha/hirag-ontology.git
%cd hirag-ontology

In [ ]:
# Set API key
import os
from google.colab import userdata

# Add your DeepSeek API key in Colab Secrets (key icon on left panel)
# Name: DEEPSEEK_API_KEY
try:
    os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
    print('API key loaded from Colab Secrets')
except:
    os.environ['DEEPSEEK_API_KEY'] = 'your_key_here'
    print('Set your DEEPSEEK_API_KEY manually')

In [ ]:
# Load pre-built knowledge graph (skip expensive extraction)
# The graph was built from 10 Minzdrav documents
import json
import sys
sys.path.insert(0, '.')

from pipeline.knowledge_graph import KnowledgeGraph

# Download pre-built graph from GitHub releases
import urllib.request
import os

os.makedirs('results', exist_ok=True)

# Use demo graph if no pre-built available
if not os.path.exists('results/knowledge_graph_final.json'):
    print('Building demo knowledge graph...')
    from evaluation.run_eval import create_demo_graph
    kg = create_demo_graph()
    kg.save('results/knowledge_graph_final.json')
else:
    kg = KnowledgeGraph.load('results/knowledge_graph_final.json')

print(f'Graph loaded: {kg.stats()}')

In [ ]:
# Run A3: Deduplication
from pipeline.deduplication import DeduplicationAgent

dedup = DeduplicationAgent(alpha=0.6, threshold=0.85)
stats = dedup.deduplicate(kg)
print(f"Deduplication: removed {stats['entities_before'] - stats['entities_after']} entities ({stats['reduction_pct']}%)")

In [ ]:
# Run A4: Validation — compute Cons(G)
from pipeline.validator import ValidationAgent

validator = ValidationAgent()
val = validator.validate(kg)
print(f"Cons(G) = {val['consistency_score']:.3f}")
print(f"Violations: {val['total_violations']}")
print(f"By type: {val['violations_by_type']}")

In [ ]:
# Compute Q(G)
from pipeline.quality import compute_quality

q = compute_quality(kg, val)
print(f"Q(G) = {q['Q']:.4f}")
print(f"Coverage = {q['coverage']:.3f}")
print(f"Consistency = {q['consistency']:.3f}")
print(f"Precision = {q['precision']:.3f}")

In [ ]:
# Run hybrid RRF retrieval
from retrieval.retriever import HybridRetriever, RetrievalMode

kg.compute_pagerank()
retriever = HybridRetriever(kg, mode=RetrievalMode.HYBRID_RRF)

query = 'What is the treatment for acute lymphoblastic leukemia?'
entity_ids = retriever.retrieve(query, top_k=10)
context = kg.get_context_for_entities(entity_ids)

print(f'Query: {query}')
print(f'Retrieved {len(entity_ids)} entities:')
for eid in entity_ids[:5]:
    e = kg.entities.get(eid)
    if e:
        print(f'  - {e.label} ({e.entity_type})')

In [ ]:
# Visualise knowledge graph with NetworkX
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

TYPE_COLORS = {
    'Condition':'#534AB7', 'Drug':'#D85A30',
    'Procedure':'#1D9E75', 'LabTest':'#185FA5',
    'AnatomicalStructure':'#378ADD', 'DosageRegimen':'#BA7517',
    'Symptom':'#639922', 'Organization':'#D4537E', 'Other':'#888780'
}

# Build nx graph
G = nx.DiGraph()
for eid, e in kg.entities.items():
    G.add_node(eid, label=e.label, etype=e.entity_type)
for r in kg.relations:
    G.add_edge(r.subject_id, r.object_id, pred=r.predicate)

# Top 25 by degree
degrees = dict(G.degree())
top25 = [n for n, _ in sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:25]]
subG = G.subgraph(top25).copy()
pos = nx.spring_layout(subG, k=2.5, seed=42)

node_colors = [TYPE_COLORS.get(G.nodes[n].get('etype','Other'),'#888') for n in subG]
node_sizes  = [degrees[n] * 18 + 150 for n in subG]
labels      = {n: G.nodes[n].get('label','')[:12] for n in subG}

fig, ax = plt.subplots(figsize=(14, 9))
nx.draw_networkx_edges(subG, pos, alpha=0.4, arrows=True, ax=ax)
nx.draw_networkx_nodes(subG, pos, node_color=node_colors,
                       node_size=node_sizes, alpha=0.9, ax=ax)
nx.draw_networkx_labels(subG, pos, labels, font_size=7,
                        font_color='white', font_weight='bold', ax=ax)

legend = [mpatches.Patch(color=c, label=t) for t, c in TYPE_COLORS.items()
          if any(G.nodes[n].get('etype') == t for n in subG)]
ax.legend(handles=legend, loc='upper left', fontsize=8)
ax.set_title('Knowledge Graph — Top 25 Entities by Degree', fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('results/colab_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graph saved to results/colab_graph.png')